In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

In [0]:
dfs = []
for periodo, tabla in [
    ('Q1_2023', 'workspace.default.listings_q_1'),
    ('Q2_2023', 'workspace.default.listings_q_2'),
    ('Q3_2023', 'workspace.default.listings_q_3'),
    ('Q4_2023', 'workspace.default.listings_q_4'),
    ('Q1_2025', 'workspace.default.listings'),
]:
    temp = spark.table(tabla).toPandas()
    temp['trimestre'] = periodo
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print(f'Filas totales: {df.shape[0]:,}')
print(f'Columnas:      {df.shape[1]}')

In [0]:
# Eliminar columnas inútiles
df = df.drop(['license', 'neighbourhood_group'], axis=1)

# Eliminar filas sin precio, sin coordenadas
df = df.dropna(subset=['price', 'latitude', 'longitude'])

# Filtrar outliers de precio
p99_price = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= p99_price)]

# Filtrar outliers de noches mínimas
df = df[df['minimum_nights'] <= 30]

print(f'Filas después de limpieza: {df.shape[0]:,}')
print(f'Columnas restantes:        {df.shape[1]}')

In [0]:
crecimiento = (
    df.groupby('trimestre')['id']
    .count()
    .reset_index()
)
crecimiento.columns = ['trimestre', 'listados']

# Orden cronológico correcto
orden = ['Q1_2023', 'Q2_2023', 'Q3_2023', 'Q4_2023', 'Q1_2025']
crecimiento['trimestre'] = pd.Categorical(crecimiento['trimestre'], categories=orden, ordered=True)
crecimiento = crecimiento.sort_values('trimestre')

plt.figure(figsize=(10, 5))
sns.lineplot(data=crecimiento, x='trimestre', y='listados', marker='o', color='steelblue')
plt.title('Crecimiento de listados Airbnb CDMX 2023-2025')
plt.xlabel('Período')
plt.ylabel('Número de listados')
plt.tight_layout()
plt.show()

La caída a Q1 2025 es el dato más relevante. Hay dos interpretaciones posibles:

Estacionalidad normal, Q1 siempre es el trimestre más bajo y el dato de 2025 simplemente refleja eso
Reducción real de la oferta, posiblemente por regulación, el debate sobre gentrificación en CDMX se intensificó en 2024 y hubo presión política sobre plataformas de renta corta

In [0]:
precio_periodo = (
    df.groupby('trimestre')['price']
    .median()
    .reset_index()
)
precio_periodo.columns = ['trimestre', 'precio_mediano']

orden = ['Q1_2023', 'Q2_2023', 'Q3_2023', 'Q4_2023', 'Q1_2025']
precio_periodo['trimestre'] = pd.Categorical(precio_periodo['trimestre'], categories=orden, ordered=True)
precio_periodo = precio_periodo.sort_values('trimestre')

plt.figure(figsize=(10, 5))
sns.lineplot(data=precio_periodo, x='trimestre', y='precio_mediano', marker='o', color='steelblue')
plt.title('Precio mediano por noche por período (MXN)')
plt.xlabel('Período')
plt.ylabel('Precio mediano (MXN)')
plt.tight_layout()
plt.show()

 El precio bajó durante 2023 cuando había más oferta, y subió significativamente a Q1 2025 cuando la oferta cayó. Eso es comportamiento de mercado clásico: menos oferta, precios más altos.

In [0]:
comercial_periodo = (
    df.groupby('trimestre')
    .apply(lambda x: (x['calculated_host_listings_count'] > 1).mean() * 100)
    .reset_index()
)
comercial_periodo.columns = ['trimestre', 'pct_comercial']

orden = ['Q1_2023', 'Q2_2023', 'Q3_2023', 'Q4_2023', 'Q1_2025']
comercial_periodo['trimestre'] = pd.Categorical(comercial_periodo['trimestre'], categories=orden, ordered=True)
comercial_periodo = comercial_periodo.sort_values('trimestre')

plt.figure(figsize=(10, 5))
sns.lineplot(data=comercial_periodo, x='trimestre', y='pct_comercial', marker='o', color='steelblue')
plt.title('Porcentaje de listados comerciales por período')
plt.xlabel('Período')
plt.ylabel('% Uso comercial')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

El uso comercial se mantiene estable alrededor del 67% durante todo 2023 y sube a ~72% en Q1 2025.
Esto significa que aunque el número total de listados bajó en 2025, los que sobrevivieron son proporcionalmente más comerciales. Los anfitriones personales salieron del mercado, los operadores comerciales se quedaron.

In [0]:
room_periodo = (
    df.groupby(['trimestre', 'room_type'])
    .size()
    .reset_index(name='listados')
)

orden = ['Q1_2023', 'Q2_2023', 'Q3_2023', 'Q4_2023', 'Q1_2025']
room_periodo['trimestre'] = pd.Categorical(room_periodo['trimestre'], categories=orden, ordered=True)
room_periodo = room_periodo.sort_values('trimestre')

plt.figure(figsize=(12, 5))
sns.barplot(data=room_periodo, x='trimestre', y='listados', hue='room_type')
plt.title('Tipos de habitación por período')
plt.xlabel('Período')
plt.ylabel('Número de listados')
plt.legend(title='Tipo', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Entire home/apt domina en todos los períodos y se mantiene estable en 2025, mientras que Private room es el que más cae. Eso es consistente con la salida de anfitriones personales del mercado.

In [0]:
calor = (
    df.groupby(['neighbourhood', 'trimestre'])
    .size()
    .reset_index(name='listados')
)

orden = ['Q1_2023', 'Q2_2023', 'Q3_2023', 'Q4_2023', 'Q1_2025']

# Top 10 alcaldías
top10 = (
    df.groupby('neighbourhood')['id']
    .count()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)

calor_top = calor[calor['neighbourhood'].isin(top10)]
calor_pivot = calor_top.pivot(index='neighbourhood', columns='trimestre', values='listados')
calor_pivot = calor_pivot[orden]

plt.figure(figsize=(12, 6))
sns.heatmap(
    calor_pivot,
    annot=True,
    fmt=',.0f',
    cmap='Blues',
    linewidths=0.5
)
plt.title('Listados por alcaldía y período (Top 10)')
plt.xlabel('Período')
plt.ylabel('')
plt.tight_layout()
plt.show()

Cuauhtémoc domina de forma absoluta en todos los períodos, más del doble que Miguel Hidalgo
La mayoría de alcaldías caen en Q1 2025, confirma la tendencia general
Hay dos excepciones que crecen: Gustavo A. Madero, Iztacalco y Venustiano Carranza aumentan en 2025, sugiere expansión hacia alcaldías más periféricas y populares
Cuauhtémoc se mantiene casi igual en 2025, es el mercado más resiliente

La expansión hacia Iztacalco y Venustiano Carranza es un dato relevante para el debate de gentrificación porque son alcaldías históricamente populares que antes no tenían presencia significativa de Airbnb.

In [0]:
precio_alcaldia = (
    df.groupby('neighbourhood')['price']
    .median()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
precio_alcaldia.columns = ['alcaldia', 'precio_mediano']

plt.figure(figsize=(12, 6))
sns.barplot(data=precio_alcaldia, x='precio_mediano', y='alcaldia', color='steelblue')
plt.title('Precio mediano por noche por alcaldía (MXN)')
plt.xlabel('Precio mediano (MXN)')
plt.ylabel('')
plt.tight_layout()
plt.show()

Miguel Hidalgo es la más cara (~$1,250 MXN), no Cuauhtémoc que tiene más listados. Eso tiene sentido porque incluye Polanco y Lomas, zonas de alto poder adquisitivo
Cuajimalpa de Morelos es la segunda más cara, incluye Santa Fe
Cuauhtémoc es tercera a pesar de tener el mayor volumen, indica que su oferta es más heterogénea
Iztapalapa es la más barata (~$450 MXN), casi un tercio del precio de Miguel Hidalgo

La conclusión clave: hay una brecha de casi 3x entre la alcaldía más cara y la más barata, lo que refleja exactamente la segregación territorial de CDMX.

In [0]:
# Listado "dedicado al turismo" = alta disponibilidad y mínimo de noches bajo
df['dedicado_turismo'] = (df['availability_365'] >= 270) & (df['minimum_nights'] <= 3)

pct_turismo = df['dedicado_turismo'].mean() * 100

print(f'Listados dedicados al turismo: {pct_turismo:.1f}%')
print(f'Listados totales en esa categoría: {df["dedicado_turismo"].sum():,}')

# Por alcaldía
turismo_alcaldia = (
    df.groupby('neighbourhood')['dedicado_turismo']
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
turismo_alcaldia.columns = ['alcaldia', 'pct_dedicado_turismo']

plt.figure(figsize=(12, 6))
sns.barplot(data=turismo_alcaldia, x='pct_dedicado_turismo', y='alcaldia', color='steelblue')
plt.title('% de listados dedicados al turismo por alcaldía (Top 10)')
plt.xlabel('% dedicado al turismo')
plt.ylabel('')
plt.tight_layout()
plt.show()

Milpa Alta lidera con ~65% de listados dedicados al turismo, siendo una alcaldía periférica y rural. Tiene sentido porque sus propiedades son principalmente cabañas y espacios naturales que se rentan por temporadas completas
Tláhuac y Xochimilco siguen con ~54%, también alcaldías periféricas con atractivos naturales y ecoturismo
Cuauhtémoc, que tiene el mayor volumen, está hasta abajo con ~48%, lo que significa que sus listados son más mixtos entre turismo y renta media

La conclusión más importante:
La presión sobre vivienda no es uniforme. Las alcaldías céntricas como Cuauhtémoc tienen más volumen pero las periféricas tienen mayor proporción de vivienda retirada del mercado tradicional. Son dos tipos distintos de impacto.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter precio vs número de reseñas
axes[0].scatter(
    df['number_of_reviews'],
    df['price'],
    alpha=0.1,
    color='steelblue',
    s=5
)
axes[0].set_title('Precio vs. número de reseñas')
axes[0].set_xlabel('Número de reseñas')
axes[0].set_ylabel('Precio (MXN)')

# Precio mediano por rango de reseñas
df['rango_reseñas'] = pd.cut(
    df['number_of_reviews'],
    bins=[-1, 0, 5, 20, 50, 100, df['number_of_reviews'].max()],
    labels=['Sin reseñas', '1-5', '6-20', '21-50', '51-100', '100+']
)

precio_reseñas = (
    df.groupby('rango_reseñas', observed=True)['price']
    .median()
    .dropna()
    .reset_index()
)

print(precio_reseñas)

plt.figure(figsize=(10, 5))
sns.barplot(data=precio_reseñas, x='rango_reseñas', y='price', color='steelblue')
plt.title('Precio mediano por rango de reseñas')
plt.xlabel('Rango de reseñas')
plt.ylabel('Precio mediano (MXN)')
plt.tight_layout()
plt.show()

Los precios son notablemente similares entre todos los rangos, entre $920 y $1,000 MXN
Las propiedades sin reseñas tienen el precio más alto (~$1,000 MXN), son nuevas o especulativas
Las propiedades con 1-5 reseñas son las más baratas (~$920 MXN), probablemente anfitriones nuevos ajustando precio para atraer clientes
No hay una correlación clara entre más reseñas y mayor precio

La conclusión: el precio en Airbnb CDMX está determinado principalmente por ubicación y tipo de propiedad, no por reputación del anfitrión. Eso refuerza que el mercado está impulsado por factores estructurales, no por calidad del servicio.

## Conclusiones

### 1. Concentración geográfica
Cuauhtémoc concentra casi el doble de listados que cualquier otra alcaldía,
seguida por Miguel Hidalgo y Benito Juárez. Estas tres alcaldías representan
la mayoría absoluta de la oferta de Airbnb en CDMX y coinciden con las zonas
de mayor presión de gentrificación en la ciudad.

### 2. Dominio del uso comercial
El 67.5% de los listados pertenece a anfitriones con más de una propiedad,
proporción que aumentó a 72% en Q1 2025. Esto indica que Airbnb en CDMX
no opera principalmente como economía colaborativa sino como mercado de
inversión inmobiliaria, con operadores que administran múltiples propiedades
de forma profesional.

### 3. Relación inversa entre oferta y precio
Durante 2023 el precio mediano bajó conforme aumentaba la oferta,
comportamiento consistente con un mercado competitivo. Sin embargo en
Q1 2025, cuando la oferta cayó por debajo de los niveles de Q1 2023,
el precio mediano subió a $1,027 MXN, su nivel más alto del período analizado.
Esto sugiere que la reducción de oferta no beneficia a los inquilinos
tradicionales sino que encarece el mercado de renta corta.

### 4. Vivienda retirada del mercado tradicional
Cerca del 50% de los listados en alcaldías periféricas como Milpa Alta,
Tláhuac y Xochimilco están dedicados exclusivamente al turismo, con alta
disponibilidad anual y mínimo de noches bajo. Esto representa vivienda
retirada del mercado de renta tradicional en zonas que históricamente
no tenían presencia significativa de plataformas de renta corta.

### 5. Expansión hacia alcaldías populares
Mientras la mayoría de alcaldías redujo su oferta en 2025, Iztacalco,
Gustavo A. Madero y Venustiano Carranza registraron crecimiento.
Esto sugiere un desplazamiento de la actividad hacia zonas
históricamente populares con precios de entrada más bajos,
un patrón consistente con la segunda fase de gentrificación documentada
en otras ciudades latinoamericanas.

### 6. Precio determinado por ubicación, no por reputación
El análisis de precio por rango de reseñas muestra diferencias menores
a $80 MXN entre listados sin reseñas y listados con más de 100.
Esto indica que el precio está determinado principalmente por
ubicación y tipo de propiedad, no por la reputación del anfitrión.

---

### Siguientes pasos
- Notebook 02: Modelo predictivo de precio con CatBoost
- Notebook 03: Serie de tiempo con datos históricos 2019-2025
  e integración de variables macroeconómicas (INPC, tipo de cambio)